# Random Forest

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import tensorflow as tf
import statsmodels.api as sm
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import pearsonr, pointbiserialr, spearmanr
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import make_scorer, mean_squared_error, r2_score, mean_absolute_error, classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve, auc, silhouette_score, precision_recall_curve, f1_score, precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, Concatenate
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [1]:
import sklearn
import tensorflow as tf

print("scikit-learn 版本：", sklearn.__version__)
print("TensorFlow 版本：", tf.__version__)

scikit-learn 版本： 1.5.2
TensorFlow 版本： 2.14.0


In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
file_path = "已知数据.xlsx"
df = pd.read_excel(file_path)

In [4]:
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗', '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]

In [5]:
y = df['标签']

In [6]:
# 连续变量
continuous_vars = ['PR', 'ki-67']

In [7]:
# 使用分层划分以保持类别比例
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [8]:
# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])

In [9]:
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [10]:
# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

In [11]:
# 定义随机森林分类器
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

In [12]:
# 重建参数网格 - 严格限制复杂度
param_grid = {
    'n_estimators': [100, 150],       # 减少树数量
    'max_depth': [3, 5, 7],           # 严格限制深度!
    'min_samples_split': [10, 20],     # 大幅提高
    'min_samples_leaf': [10, 20],      # 最低10个样本
    'max_features': ['sqrt'],          # 固定推荐值
    'class_weight': [None, 'balanced', class_weight_dict]    # 强制类别平衡
}

In [13]:
# 创建F1评估器
f1_scorer = make_scorer(f1_score, pos_label=1)

In [14]:
# 使用网格搜索和交叉验证来寻找最佳参数
#grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='roc_auc')
grid_search = GridSearchCV(
                    estimator=rf,
                    param_grid=param_grid,
                    scoring=f1_scorer,  # 关键修改！用F1替代准确率
                    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                    n_jobs=-1
)

In [15]:
# 拟合模型
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(class_weight='balanced',
                                              random_state=42),
             n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced',
                                          {0: 0.6183431952662722, 1: 2.6125}],
                         'max_depth': [3, 5, 7], 'max_features': ['sqrt'],
                         'min_samples_leaf': [10, 20],
                         'min_samples_split': [10, 20],
                         'n_estimators': [100, 150]},
             scoring=make_scorer(f1_score, response_method='predict', pos_label=1))

In [16]:
# 在测试集上评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 输出评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'class_weight': {0: 0.6183431952662722, 1: 2.6125}, 'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 20, 'min_samples_split': 10, 'n_estimators': 100}
测试集准确率 Accuracy: 0.6667
测试集召回率 Recall: 0.2917
测试集F1分数: 0.2500
测试集AUC值: 0.6111

测试集分类报告：
               precision    recall  f1-score   support

           0       0.82      0.75      0.79       102
           1       0.22      0.29      0.25        24

    accuracy                           0.67       126
   macro avg       0.52      0.52      0.52       126
weighted avg       0.70      0.67      0.68       126



In [17]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("rf_roc_curve.csv", index=False)
print("ROC数据已保存为 rf_roc_curve.csv")

ROC数据已保存为 rf_roc_curve.csv


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer

# 设置随机种子
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

# 特征与标签
X = df[['PR', '腋窝淋巴结状态', '手术前怀孕', '目前月经情况', 'ki-67', 'HER2+FISH', '治疗后怀孕',
        '治疗后生产', 'LN转移个数', '放疗', '化疗期间是否应用诺雷德',
        '靶向治疗（赫赛汀或赫赛汀+帕捷特）', '手术方式', '化疗方案', '内分泌治疗方案']]
y = df['标签']

# 连续变量标准化
continuous_vars = ['PR', 'ki-67']

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 2. 连续变量：进行标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])

X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# 定义随机森林
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# 参数网格
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 5, 7],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [10, 20],
    'max_features': ['sqrt'],
    'class_weight': [None, 'balanced', class_weight_dict]
}

# F1 scorer
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索 + 交叉验证（在训练集上）
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# 测试集评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 输出评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'class_weight': 'balanced', 'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 100}
测试集准确率 Accuracy: 0.6667
测试集召回率 Recall: 0.5417
测试集F1分数: 0.3824
测试集AUC值: 0.7124

测试集分类报告：
               precision    recall  f1-score   support

           0       0.87      0.70      0.77       102
           1       0.30      0.54      0.38        24

    accuracy                           0.67       126
   macro avg       0.58      0.62      0.58       126
weighted avg       0.76      0.67      0.70       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("rf_bjd_roc_curve.csv", index=False)
print("ROC数据已保存为 rf_bjd_roc_curve.csv")

ROC数据已保存为 rf_bjd_roc_curve.csv


# XGBoost

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保结果可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "已知数据.xlsx"
df = pd.read_excel(file_path)

# 特征和标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗', 
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 分层划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 标准化连续变量
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重（XGBoost 不直接支持 class_weight，但我们可以通过 scale_pos_weight 参数近似实现）
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# 如果是二分类：类别为0和1，设置 scale_pos_weight = neg/pos
pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义XGBoost分类器
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# 参数网格（根据XGBoost调参经验设置）
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'min_child_weight': [1, 5],
    'scale_pos_weight': [pos_weight],  # 加入类别不平衡处理
}

# F1作为评估指标
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 模型训练
grid_search.fit(X_train, y_train)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 150, 'scale_pos_weight': 0.23668639053254442, 'subsample': 0.8}
测试集准确率 Accuracy: 0.7937
测试集召回率 Recall: 0.0417
测试集F1分数: 0.0714
测试集AUC值: 0.6462

测试集分类报告：
               precision    recall  f1-score   support

           0       0.81      0.97      0.88       102
           1       0.25      0.04      0.07        24

    accuracy                           0.79       126
   macro avg       0.53      0.51      0.48       126
weighted avg       0.70      0.79      0.73       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("xgb_roc_curve.csv", index=False)
print("ROC数据已保存为 xgb_roc_curve.csv")

ROC数据已保存为 xgb_roc_curve.csv


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保结果可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

# 特征和标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗', 
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 标准化连续变量
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重（XGBoost 不直接支持 class_weight，但我们可以通过 scale_pos_weight 参数近似实现）
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# 如果是二分类：类别为0和1，设置 scale_pos_weight = neg/pos
pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义XGBoost分类器
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# 参数网格（根据XGBoost调参经验设置）
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'min_child_weight': [1, 5],
    'scale_pos_weight': [pos_weight],  # 加入类别不平衡处理
}

# F1作为评估指标
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 模型训练
grid_search.fit(X_train, y_train)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 150, 'scale_pos_weight': 0.5810185185185186, 'subsample': 0.8}
测试集准确率 Accuracy: 0.7540
测试集召回率 Recall: 0.5000
测试集F1分数: 0.4364
测试集AUC值: 0.6977

测试集分类报告：
               precision    recall  f1-score   support

           0       0.87      0.81      0.84       102
           1       0.39      0.50      0.44        24

    accuracy                           0.75       126
   macro avg       0.63      0.66      0.64       126
weighted avg       0.78      0.75      0.77       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("xgb_bjd_roc_curve.csv", index=False)
print("ROC数据已保存为 xgb_bjd_roc_curve.csv")

ROC数据已保存为 xgb_bjd_roc_curve.csv


# Gradient Boosting Trees

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保结果可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "已知数据.xlsx"
df = pd.read_excel(file_path)

# 特征和标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗', 
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 分层划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 标准化连续变量
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# 计算正类/负类样本比例（手动调整 sample_weight 时可能用得到）
pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义梯度提升分类器
gbdt = GradientBoostingClassifier(random_state=42)

# 设置参数网格
param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [10, 20],
    'subsample': [0.8, 1.0]
}

# F1评分函数
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索交叉验证
grid_search = GridSearchCV(
    estimator=gbdt,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 拟合模型
grid_search.fit(X_train, y_train)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# 输出结果
print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 150, 'subsample': 1.0}
测试集准确率 Accuracy: 0.7460
测试集召回率 Recall: 0.2083
测试集F1分数: 0.2381
测试集AUC值: 0.6095

测试集分类报告：
               precision    recall  f1-score   support

           0       0.82      0.87      0.85       102
           1       0.28      0.21      0.24        24

    accuracy                           0.75       126
   macro avg       0.55      0.54      0.54       126
weighted avg       0.72      0.75      0.73       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("gbt_roc_curve.csv", index=False)
print("ROC数据已保存为 gbt_roc_curve.csv")

ROC数据已保存为 gbt_roc_curve.csv


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保结果可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

# 特征和标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗', 
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 标准化连续变量
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# 计算正类/负类样本比例（手动调整 sample_weight 时可能用得到）
pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义梯度提升分类器
gbdt = GradientBoostingClassifier(random_state=42)

# 设置参数网格
param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [10, 20],
    'subsample': [0.8, 1.0]
}

# F1评分函数
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索交叉验证
grid_search = GridSearchCV(
    estimator=gbdt,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 拟合模型
grid_search.fit(X_train, y_train)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 评估指标
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# 输出结果
print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'learning_rate': 0.05, 'max_depth': 5, 'min_samples_leaf': 10, 'min_samples_split': 10, 'n_estimators': 100, 'subsample': 0.8}
测试集准确率 Accuracy: 0.7381
测试集召回率 Recall: 0.5417
测试集F1分数: 0.4407
测试集AUC值: 0.7083

测试集分类报告：
               precision    recall  f1-score   support

           0       0.88      0.78      0.83       102
           1       0.37      0.54      0.44        24

    accuracy                           0.74       126
   macro avg       0.63      0.66      0.63       126
weighted avg       0.78      0.74      0.76       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("gbt_bjd_roc_curve.csv", index=False)
print("ROC数据已保存为 gbt_bjd_roc_curve.csv")

ROC数据已保存为 gbt_bjd_roc_curve.csv


# LightGBM

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "已知数据.xlsx"
df = pd.read_excel(file_path)

# 特征与标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗',
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 分层划分训练集与测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 连续变量标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重（正类比例调整）
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# LightGBM 用 scale_pos_weight 来处理类别不平衡（仅对二分类有效）
scale_pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义 LGBM 分类器
lgbm = LGBMClassifier(
    random_state=42,
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    boosting_type='gbdt',
    n_jobs=-1,
    verbose=-1
)

# 设置参数搜索网格
param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'num_leaves': [15, 31],  # 和 max_depth 联动
    'min_child_samples': [10, 20],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# F1 分数作为调参指标
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 模型训练
grid_search.fit(X_train, y_train)

# 最佳模型预测
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 模型评估
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# 输出结果
print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 150, 'num_leaves': 31, 'subsample': 0.8}
测试集准确率 Accuracy: 0.7778
测试集召回率 Recall: 0.1667
测试集F1分数: 0.2222
测试集AUC值: 0.6348

测试集分类报告：
               precision    recall  f1-score   support

           0       0.82      0.92      0.87       102
           1       0.33      0.17      0.22        24

    accuracy                           0.78       126
   macro avg       0.58      0.54      0.55       126
weighted avg       0.73      0.78      0.75       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("lgbm_roc_curve.csv", index=False)
print("ROC数据已保存为 lgbm_roc_curve.csv")

ROC数据已保存为 lgbm_roc_curve.csv


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子以确保可重复
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

# 特征与标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗',
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 连续变量标准化
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 计算类别权重（正类比例调整）
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
# LightGBM 用 scale_pos_weight 来处理类别不平衡（仅对二分类有效）
scale_pos_weight = class_weight_dict[0] / class_weight_dict[1]

# 定义 LGBM 分类器
lgbm = LGBMClassifier(
    random_state=42,
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    boosting_type='gbdt',
    n_jobs=-1,
    verbose=-1
)

# 设置参数搜索网格
param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7],
    'num_leaves': [15, 31],  # 和 max_depth 联动
    'min_child_samples': [10, 20],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# F1 分数作为调参指标
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 模型训练
grid_search.fit(X_train, y_train)

# 最佳模型预测
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 模型评估
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# 输出结果
print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_samples': 10, 'n_estimators': 100, 'num_leaves': 15, 'subsample': 0.8}
测试集准确率 Accuracy: 0.7778
测试集召回率 Recall: 0.5000
测试集F1分数: 0.4615
测试集AUC值: 0.7177

测试集分类报告：
               precision    recall  f1-score   support

           0       0.88      0.84      0.86       102
           1       0.43      0.50      0.46        24

    accuracy                           0.78       126
   macro avg       0.65      0.67      0.66       126
weighted avg       0.79      0.78      0.78       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)


roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("lgbm_bjd_roc_curve.csv", index=False)
print("ROC数据已保存为 lgbm_bjd_roc_curve.csv")

ROC数据已保存为 lgbm_bjd_roc_curve.csv


# CatBoost

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "已知数据.xlsx"
df = pd.read_excel(file_path)

# 特征与标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗',
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 类别变量索引（CatBoost支持类别型变量索引列表）
categorical_features = [i for i, col in enumerate(X.columns) if col not in continuous_vars]

# 分层划分训练测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 标准化连续变量（注意：CatBoost支持原始输入，但标准化仍然可提升性能）
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 类别权重：CatBoost支持 class_weights 参数
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# 定义 CatBoost 分类器（设置 silent 和 class_weights）
catboost = CatBoostClassifier(
    random_state=42,
    verbose=0,
    class_weights=class_weight_dict,
    loss_function='Logloss',
    eval_metric='F1',
    task_type='CPU'
)

# 参数网格
param_grid = {
    'iterations': [100, 200],
    'learning_rate': [0.01, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5]
}

# F1 评分器
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 拟合模型（指定类别特征索引）
grid_search.fit(X_train, y_train, cat_features=categorical_features)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 输出评估结果
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'depth': 4, 'iterations': 200, 'l2_leaf_reg': 1, 'learning_rate': 0.01}
测试集准确率 Accuracy: 0.6111
测试集召回率 Recall: 0.5000
测试集F1分数: 0.3288
测试集AUC值: 0.6001

测试集分类报告：
               precision    recall  f1-score   support

           0       0.84      0.64      0.73       102
           1       0.24      0.50      0.33        24

    accuracy                           0.61       126
   macro avg       0.54      0.57      0.53       126
weighted avg       0.73      0.61      0.65       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("catb_roc_curve.csv", index=False)
print("ROC数据已保存为 catb_roc_curve.csv")

ROC数据已保存为 catb_roc_curve.csv


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, classification_report, make_scorer
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子
np.random.seed(42)
tf.random.set_seed(42)

# 加载数据
file_path = "预测.xlsx"
df = pd.read_excel(file_path)

# 特征与标签
X = df[['PR','腋窝淋巴结状态','手术前怀孕','目前月经情况','ki-67','HER2+FISH','治疗后怀孕','治疗后生产','LN转移个数','放疗',
        '化疗期间是否应用诺雷德', '靶向治疗（赫赛汀或赫赛汀+帕捷特）','手术方式','化疗方案','内分泌治疗方案']]
y = df['标签']

# 连续变量
continuous_vars = ['PR', 'ki-67']

# 类别变量索引（CatBoost支持类别型变量索引列表）
categorical_features = [i for i, col in enumerate(X.columns) if col not in continuous_vars]

# 从前627行中划分出20%作为测试集
X_front = X.iloc[:627]
y_front = y.iloc[:627]
X_front_train, X_test, y_front_train, y_test = train_test_split(
    X_front, y_front, test_size=0.2, stratify=y_front, random_state=42
)

# 剩下的数据（从第628行开始）作为训练集的一部分
X_rest = X.iloc[627:]
y_rest = y.iloc[627:]

# 拼接训练数据：前627行的80% + 剩下所有行
X_train = pd.concat([X_front_train, X_rest], axis=0)
y_train = pd.concat([y_front_train, y_rest], axis=0)

# 标准化连续变量（注意：CatBoost支持原始输入，但标准化仍然可提升性能）
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])
X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

# 类别权重：CatBoost支持 class_weights 参数
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# 定义 CatBoost 分类器（设置 silent 和 class_weights）
catboost = CatBoostClassifier(
    random_state=42,
    verbose=0,
    class_weights=class_weight_dict,
    loss_function='Logloss',
    eval_metric='F1',
    task_type='CPU'
)

# 参数网格
param_grid = {
    'iterations': [100, 200],
    'learning_rate': [0.01, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5]
}

# F1 评分器
f1_scorer = make_scorer(f1_score, pos_label=1)

# 网格搜索
grid_search = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

# 拟合模型（指定类别特征索引）
grid_search.fit(X_train, y_train, cat_features=categorical_features)

# 最佳模型评估
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

# 输出评估结果
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("最佳参数：", grid_search.best_params_)
print(f"测试集准确率 Accuracy: {accuracy:.4f}")
print(f"测试集召回率 Recall: {recall:.4f}")
print(f"测试集F1分数: {f1:.4f}")
print(f"测试集AUC值: {auc:.4f}")
print("\n测试集分类报告：\n", classification_report(y_test, y_pred))

最佳参数： {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.1}
测试集准确率 Accuracy: 0.7063
测试集召回率 Recall: 0.5000
测试集F1分数: 0.3934
测试集AUC值: 0.6671

测试集分类报告：
               precision    recall  f1-score   support

           0       0.87      0.75      0.81       102
           1       0.32      0.50      0.39        24

    accuracy                           0.71       126
   macro avg       0.59      0.63      0.60       126
weighted avg       0.76      0.71      0.73       126



In [2]:
# 导入ROC曲线函数
from sklearn.metrics import roc_curve

# 计算FPR, TPR, 阈值
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# 可保存为文件用于后续绘图
roc_df = pd.DataFrame({'FPR': fpr, 'TPR': tpr, 'Threshold': thresholds})
roc_df.to_csv("catb_bjd_roc_curve.csv", index=False)
print("ROC数据已保存为 catb_bjd_roc_curve.csv")

ROC数据已保存为 catb_bjd_roc_curve.csv
